# Stage 4 — Model training and OOF prediction

This notebook is the single Stage-4 orchestration entry point. The reusable implementations live in `src/models/`; the notebook loads protected artifacts, runs each model, and displays aggregate diagnostics.

> **PhysioNet DUA:** run only in the controlled Kaggle/Colab environment containing the protected Stage-2/3 artifacts. Never display patient rows or identifiers. OOF predictions and fitted models must remain in the gitignored `data/` and `results/` directories.

## Stage-4 protocol

1. Reuse the frozen 20% internal holdout and five development folds from Stage 3.
2. Fit imputation and numeric scaling on original training-fold rows, then run SMOTENC, then one-hot encode categories. Never refit a scaler on synthetic rows. Class weights are also training-fold only.
3. Select hyperparameters by mean development-fold AUROC; use mean AUPRC as the tie-breaker.
4. Produce exactly one out-of-fold probability for every development row.
5. Refit the selected configuration on the complete development set.
6. Do not fit, predict, evaluate, or tune against the internal test partition in this stage.
7. Leave F1-threshold selection and final model comparison to Stage 5.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DATA_PROCESSED, N_CV_FOLDS, RANDOM_SEED
from src.models.classic import (
    logistic_regression_candidates,
    load_static_stage4_inputs,
    mlp_candidates,
    save_static_training_result,
    svm_candidates,
    train_logistic_regression,
    train_mlp,
    train_svm,
    train_xgboost,
    xgboost_candidates,
)
from src.resampling_audit import audit_smotenc_training_fold

# Version the corrected run so the previous screening artifacts remain intact.
ARTIFACT_SUFFIX = 'smotenc_scaled_v2'

np.random.seed(RANDOM_SEED)
print(f'Project root: {ROOT}')
print(f'Frozen CV folds: {N_CV_FOLDS}; random seed: {RANDOM_SEED}')

## 1. Load and revalidate protected Stage-2/3 artifacts

The loader checks row alignment, frozen assignments, all five folds, and patient separation. This cell prints aggregate counts only.

In [ ]:
static, splits = load_static_stage4_inputs()
assignments = splits.assignments
dev = assignments['split'].eq('dev')
test = assignments['split'].eq('test')

assert len(splits.dev_indices) + len(splits.test_indices) == len(static)
assert set(assignments.loc[dev, 'cv_fold']) == set(range(N_CV_FOLDS))
assert set(assignments.loc[dev, 'subject_id']).isdisjoint(
    set(assignments.loc[test, 'subject_id'])
)

print(f'Total rows: {len(static):,}')
print(f'Development rows: {dev.sum():,}')
print(f'Internal-test rows (sealed): {test.sum():,}')
print(f'Development prevalence: {assignments.loc[dev, "label"].mean():.3%}')

### SMOTENC correction and optional quality audit

Restart the kernel after updating the source, then rerun this notebook with the existing Stage-2 features and frozen Stage-3 assignments; do not regenerate the splits. Both LR and XGBoost now use training-only numeric scaling before SMOTENC. Even `scale_numeric=False` cannot disable this distance-scaling requirement. Validation/prediction uses that same fitted scaler and never resamples.

The optional audit below separately resamples only the first fold's training partition and prints aggregate synthetic-quality counts. It does not train a model, touch validation/test rows, or change the screening. Check the reported denominators: missing features mean a check is unavailable, not passed. Fractional counts describe feature-space interpolation; contradictory missing flags/counts are a separate warning. No counts are rounded and no missing flags are silently repaired. This is a spot check, not proof that every synthetic sample or fold is clinically plausible.

The corrected run uses the `smotenc_scaled_v2` artifact suffix (plus `_smoke` for smoke runs), preserving the old unsuffixed and `_smoke` artifacts. Repeating the same version still replaces its own artifacts.

In [ ]:
RUN_SMOTENC_QUALITY_AUDIT = False

if RUN_SMOTENC_QUALITY_AUDIT:
    audit_train_indices, _ = next(splits.iter_cv())
    audit_rows = []
    for ratio in (0.10, 0.25, 0.50, 1.00):
        audit_rows.append({
            'fold': 0,
            'sampling_strategy': ratio,
            **audit_smotenc_training_fold(
                static.iloc[audit_train_indices],
                sampling_strategy=ratio,
                random_state=RANDOM_SEED,
            ),
        })
    display(pd.DataFrame(audit_rows))
else:
    print('Optional training-fold quality audit is off; set RUN_SMOTENC_QUALITY_AUDIT=True to run.')

## 2. Inspect the imbalance-screening space

The smoke profile has three configurations and validates baseline, fold-derived class weighting, and SMOTENC on all five folds. The screening profile fixes LR at `C=1`, L2 and compares six mutually exclusive imbalance strategies, isolating the effect of imbalance handling. SMOTENC targets minority/majority ratios of 0.10, 0.25, 0.50, or 1.00 and runs before one-hot encoding; it is never combined with class weighting. Full LR hyperparameter tuning is intentionally deferred until the strategy shortlist is fixed.

In [ ]:
lr_smoke_candidates = logistic_regression_candidates(profile='smoke')
lr_screening_candidates = logistic_regression_candidates(profile='screening')
print(f'LR Smoke candidates: {len(lr_smoke_candidates)}')
print(f'LR Imbalance-screening candidates: {len(lr_screening_candidates)}')
display(pd.DataFrame([candidate.as_dict() for candidate in lr_screening_candidates]))

In [ ]:
xgb_smoke_candidates = xgboost_candidates(profile="smoke")
xgb_screening_candidates = xgboost_candidates(profile="screening")
print(f'XGBoost Smoke candidates: {len(xgb_smoke_candidates)}')
print(f'XGBoost Imbalance-screening candidates: {len(xgb_screening_candidates)}')
display(pd.DataFrame([candidate.as_dict() for candidate in xgb_screening_candidates]))

## 3. Smoke-run

This still uses all five frozen development folds, comparing baseline, fold-derived class weighting, and SMOTENC 0.25 for each model. LR uses `C=1`, L2. Smoke artifacts receive the version suffix plus `_smoke` and cannot overwrite the screening run.

In [ ]:
lr_smoke_result = train_logistic_regression(
    static,
    splits,
    profile='smoke',
    progress_callback=print,
)
lr_smoke_artifacts = save_static_training_result(
    lr_smoke_result, artifact_suffix=f'{ARTIFACT_SUFFIX}_smoke'
)
display(lr_smoke_result.strategy_metrics)
print(f'LR smoke best candidate: {lr_smoke_result.best_candidate.name}')
print(f'LR protected smoke OOF: {lr_smoke_artifacts.oof_path}')
print(f'LR protected per-strategy OOF: {lr_smoke_artifacts.strategy_oof_path}')

In [ ]:
xgb_smoke_result = train_xgboost(
    static,
    splits,
    profile="smoke",
    device="cuda",
    progress_callback=print,
)
xgb_smoke_artifacts = save_static_training_result(
    xgb_smoke_result, artifact_suffix=f'{ARTIFACT_SUFFIX}_smoke'
)
display(xgb_smoke_result.strategy_metrics)
print(f'XGBoost smoke best candidate: {xgb_smoke_result.best_candidate.name}')
print(f'XGBoost protected smoke OOF: {xgb_smoke_artifacts.oof_path}')
print(f'XGBoost protected per-strategy OOF: {xgb_smoke_artifacts.strategy_oof_path}')

## 4. Run the imbalance screening

Run this only after the smoke result completes and the assertions below remain clean. Change each model's gate to `True`; each screening performs 6 × 5 fitted pipelines across six strategies, then refits its winner on dev. Artifacts use `ARTIFACT_SUFFIX`, preserving the previous pipeline's results.

In [ ]:
RUN_LR_IMBALANCE_SCREENING = False

if RUN_LR_IMBALANCE_SCREENING:
    lr_result = train_logistic_regression(
        static,
        splits,
        profile='screening',
        progress_callback=print,
    )
    lr_artifacts = save_static_training_result(lr_result, artifact_suffix=ARTIFACT_SUFFIX)
    display(lr_result.strategy_metrics)
    print(f'Final LR candidate: {lr_result.best_candidate.name}')
    print(f'Protected LR OOF: {lr_artifacts.oof_path}')
    print(f'LR protected per-strategy OOF: {lr_artifacts.strategy_oof_path}')
    print(f'LR fitted development model: {lr_artifacts.model_path}')
else:
    print('LR screening is gated. Set RUN_LR_IMBALANCE_SCREENING=True when ready.')

In [ ]:
RUN_XGB_IMBALANCE_SCREENING = False

if RUN_XGB_IMBALANCE_SCREENING:
    xgb_result = train_xgboost(
        static,
        splits,
        profile="screening",
        device="cuda",
        progress_callback=print,
    )
    xgb_artifacts = save_static_training_result(xgb_result, artifact_suffix=ARTIFACT_SUFFIX)
    display(xgb_result.strategy_metrics)
    print(f'Final XGBoost candidate: {xgb_result.best_candidate.name}')
    print(f'Protected XGBoost OOF: {xgb_artifacts.oof_path}')
    print(f'XGBoost protected per-strategy OOF: {xgb_artifacts.strategy_oof_path}')
    print(f'XGBoost fitted development model: {xgb_artifacts.model_path}')

## 5. Leakage and completeness audit

The assertions apply to the final result when available, otherwise to the smoke run. They confirm that every development row has one finite OOF probability and that no internal-test row appears in the OOF artifact.

In [ ]:
checked_result = lr_result if RUN_LR_IMBALANCE_SCREENING else lr_smoke_result
oof = checked_result.oof_predictions

assert len(oof) == len(splits.dev_indices)
assert oof['row_index'].is_unique
assert set(oof['row_index']) == set(splits.dev_indices)
assert set(oof['row_index']).isdisjoint(splits.test_indices)
assert np.isfinite(oof['probability']).all()
assert oof['probability'].between(0, 1).all()
assert set(oof['cv_fold']) == set(range(N_CV_FOLDS))
strategy_oof = checked_result.strategy_oof_predictions
n_strategies = checked_result.strategy_metrics['strategy'].nunique()
assert len(strategy_oof) == len(splits.dev_indices) * n_strategies
assert strategy_oof.groupby('strategy')['row_index'].nunique().eq(len(splits.dev_indices)).all()
assert set(strategy_oof['row_index']).isdisjoint(splits.test_indices)

print(f'LR global-best OOF rows verified: {len(oof):,}')
print(f'LR per-strategy OOF rows verified: {len(strategy_oof):,}')
print('LR internal test remains sealed: no Stage-4 predictions were created for it.')

In [ ]:
checked_result = xgb_result if RUN_XGB_IMBALANCE_SCREENING else xgb_smoke_result
oof = checked_result.oof_predictions

assert len(oof) == len(splits.dev_indices)
assert oof['row_index'].is_unique
assert set(oof['row_index']) == set(splits.dev_indices)
assert set(oof['row_index']).isdisjoint(splits.test_indices)
assert np.isfinite(oof['probability']).all()
assert oof['probability'].between(0, 1).all()
assert set(oof['cv_fold']) == set(range(N_CV_FOLDS))
strategy_oof = checked_result.strategy_oof_predictions
n_strategies = checked_result.strategy_metrics['strategy'].nunique()
assert len(strategy_oof) == len(splits.dev_indices) * n_strategies
assert strategy_oof.groupby('strategy')['row_index'].nunique().eq(len(splits.dev_indices)).all()
assert set(strategy_oof['row_index']).isdisjoint(splits.test_indices)

print(f'XGBoost global-best OOF rows verified: {len(oof):,}')
print(f'XGBoost per-strategy OOF rows verified: {len(strategy_oof):,}')
print('XGBoost internal test remains sealed: no Stage-4 predictions were created for it.')

## 6. SVM: grouped probability calibration, smoke, then tuning

SVM keeps the same five frozen outer development folds. Inside each outer training partition, three patient-grouped folds produce held-out SVM decision scores for sigmoid calibration. Every inner fit builds its own preprocessing, computes its own class weight, and runs SMOTENC only on its own training rows; calibration rows keep their natural prevalence. The calibrated model then refits its base SVM on the whole outer training partition. The outer validation partition and sealed internal test never enter that calibration fit. These inner folds calibrate probabilities; they do not perform another hyperparameter search.

`SVC` runs on CPU: do not pass `device` or `probability=True`. Each outer fit contains three inner fits plus one base-model refit. A complete fresh smoke run therefore uses `3 × 5 × 4 + 4 = 64` base SVM fits, including the final winner's development refit. Measure smoke runtime before enabling tuning; the default 36-candidate tuning run uses 724 base fits without cache reuse. Both gates start off.

Smoke has three strategies (baseline, class weight, SMOTENC 0.25) and no checkpoint. Tuning crosses 12 kernel/C/gamma configurations with baseline, class weight, and SMOTENC 0.10. Numeric scaling is enabled for every SVM candidate. Saved models include probability calibration and provide `predict_proba()`. The following cells show the candidate configurations, run gates, and restart precautions.

In [ ]:
svm_smoke_candidates = svm_candidates(profile='smoke')
svm_tuning_candidates = svm_candidates(profile='tuning')
print(f'SVM Smoke candidates: {len(svm_smoke_candidates)}')
print(f'SVM Tuning candidates: {len(svm_tuning_candidates)}')
display(pd.DataFrame([candidate.as_dict() for candidate in svm_tuning_candidates]))

# Keep new calibration runs separate from previous model artifacts/checkpoints.
SVM_ARTIFACT_SUFFIX = 'grouped_calibration_v1'
SVM_CHECKPOINT_DIR = DATA_PROCESSED / 'checkpoints' / f'svm_{SVM_ARTIFACT_SUFFIX}'

In [ ]:
RUN_SVM_SMOKE = False
svm_smoke_result = None

if RUN_SVM_SMOKE:
    svm_smoke_result = train_svm(
        static, splits, profile='smoke',
        checkpoint_dir=None, progress_callback=print,
    )
    svm_smoke_artifacts = save_static_training_result(
        svm_smoke_result, artifact_suffix=f'{SVM_ARTIFACT_SUFFIX}_smoke'
    )
    display(svm_smoke_result.strategy_metrics)
    print(f'SVM smoke best candidate: {svm_smoke_result.best_candidate.name}')
    print(f'SVM protected smoke OOF: {svm_smoke_artifacts.oof_path}')
    print(f'SVM protected per-strategy OOF: {svm_smoke_artifacts.strategy_oof_path}')
else:
    print('SVM smoke is gated. Set RUN_SVM_SMOKE=True when ready.')

### Enable SVM tuning only after the smoke run succeeds

Use the same code, dependency versions, candidates, frozen splits, and directory to resume. Checkpoints are still per outer candidate/fold: a crash during one of its inner fits reruns that unfinished outer fold; completed outer folds are reusable. `checkpoint_dir=None` disables checkpointing; `resume=False` does not mean disable or overwrite.

This SVM implementation changes the shared `classic.py` source fingerprint. Previous unfinished LR/XGBoost/RF checkpoints can therefore reject the new code as well. Finish those runs with their matching code, or use a fresh directory for a new experiment. Do not edit manifests to bypass validation. Existing saved results are not deleted, and Stage-2 features/Stage-3 splits do not need regenerating.

Checkpoint files, fitted models, and OOF predictions are protected derivatives. Keep them in the controlled environment and never publish notebook outputs or upload these files to a third-party service. Checkpointing only helps if its files survive the Kaggle/Colab session.

In [ ]:
RUN_SVM_TUNING = False
svm_tuning_result = None

if RUN_SVM_TUNING:
    svm_tuning_result = train_svm(
        static, splits, profile='tuning',
        checkpoint_dir=SVM_CHECKPOINT_DIR,
        resume=True, progress_callback=print,
    )
    svm_tuning_artifacts = save_static_training_result(
        svm_tuning_result, artifact_suffix=f'{SVM_ARTIFACT_SUFFIX}_tuning'
    )
    display(svm_tuning_result.strategy_metrics)
    print(f'Final SVM candidate: {svm_tuning_result.best_candidate.name}')
    print(f'SVM protected OOF: {svm_tuning_artifacts.oof_path}')
    print(f'SVM protected per-strategy OOF: {svm_tuning_artifacts.strategy_oof_path}')
    print(f'SVM fitted development model: {svm_tuning_artifacts.model_path}')
else:
    print('SVM tuning is gated. Set RUN_SVM_TUNING=True after checking smoke.')

In [ ]:
svm_checked_result = (
    svm_tuning_result if svm_tuning_result is not None else svm_smoke_result
)
if svm_checked_result is not None:
    svm_oof = svm_checked_result.oof_predictions
    assert len(svm_oof) == len(splits.dev_indices)
    assert svm_oof['row_index'].is_unique
    assert set(svm_oof['row_index']) == set(splits.dev_indices)
    assert set(svm_oof['row_index']).isdisjoint(splits.test_indices)
    assert np.isfinite(svm_oof['probability']).all()
    assert svm_oof['probability'].between(0, 1).all()
    assert set(svm_oof['cv_fold']) == set(range(N_CV_FOLDS))
    svm_strategy_oof = svm_checked_result.strategy_oof_predictions
    n_svm_strategies = svm_checked_result.strategy_metrics['strategy'].nunique()
    assert len(svm_strategy_oof) == len(splits.dev_indices) * n_svm_strategies
    assert not svm_strategy_oof.duplicated(['strategy', 'row_index']).any()
    assert svm_strategy_oof.groupby('strategy')['row_index'].nunique().eq(len(splits.dev_indices)).all()
    assert set(svm_strategy_oof['row_index']) == set(splits.dev_indices)
    assert np.isfinite(svm_strategy_oof['probability']).all()
    assert svm_strategy_oof['probability'].between(0, 1).all()
    print(f'SVM global-best OOF rows verified: {len(svm_oof):,}')
    print(f'SVM per-strategy OOF rows verified: {len(svm_strategy_oof):,}')
    print('SVM internal test remains sealed: no Stage-4 predictions were created for it.')
else:
    print('SVM OOF audit skipped: no SVM run has completed in this session.')

## 7. PyTorch MLP: grouped early stopping, smoke, then tuning

MLP uses the same static features, numeric scaling, and five frozen outer development folds. Inside each outer training partition, a patient-grouped holdout of approximately 20% selects the training epoch with the best inner AUROC. Preprocessing is fitted on inner training rows; SMOTENC and positive-class weighting also use only that training subset. The model then starts afresh and refits preprocessing and the network on the whole outer training partition for exactly the selected number of epochs. The final development refit follows the same procedure. Neither the outer validation partition nor the sealed internal test selects an epoch. This inner split is for early stopping, not SVM-style probability calibration.

The network uses ReLU, dropout, AdamW, and `BCEWithLogitsLoss`; inference uses sigmoid and evaluation mode. The cost-sensitive strategy supplies a positive-class loss weight computed from the original rows of each actual training subset. SMOTENC and class weighting remain separate strategies. Fitted predictors are retained on CPU for portable saving and inference; `MLP_DEVICE` controls training.

Smoke has three candidates and a five-epoch maximum so it checks integration and runtime, not model quality. Tuning draws 10 network configurations and crosses them with baseline, class weight, and SMOTENC 0.10, giving 30 candidates. Check the smoke run before enabling tuning. The candidate table below lists the network settings; the following sections explain training diagnostics and recovery.

In [ ]:
import torch

MLP_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'MLP training device: {MLP_DEVICE}; PyTorch: {torch.__version__}')
mlp_smoke_candidates = mlp_candidates(profile='smoke')
mlp_tuning_candidates = mlp_candidates(profile='tuning')
print(f'MLP Smoke candidates: {len(mlp_smoke_candidates)}')
print(f'MLP Tuning candidates: {len(mlp_tuning_candidates)}')
display(pd.DataFrame([candidate.as_dict() for candidate in mlp_tuning_candidates]))

MLP_ARTIFACT_SUFFIX = 'torch_v1'
MLP_CHECKPOINT_DIR = DATA_PROCESSED / 'checkpoints' / f'mlp_{MLP_ARTIFACT_SUFFIX}'

In [ ]:
RUN_MLP_SMOKE = False
mlp_smoke_result = None

if RUN_MLP_SMOKE:
    mlp_smoke_result = train_mlp(
        static, splits, profile='smoke', device=MLP_DEVICE,
        checkpoint_dir=None, progress_callback=print,
    )
    mlp_smoke_artifacts = save_static_training_result(
        mlp_smoke_result, artifact_suffix=f'{MLP_ARTIFACT_SUFFIX}_smoke'
    )
    display(mlp_smoke_result.strategy_metrics)
    display(mlp_smoke_result.fold_metrics[[
        'candidate', 'fold', 'selected_epochs', 'selection_epochs',
        'inner_validation_auroc', 'fit_seconds',
    ]])
    print(f'MLP smoke best candidate: {mlp_smoke_result.best_candidate.name}')
    print(f'MLP protected smoke OOF: {mlp_smoke_artifacts.oof_path}')
    print(f'MLP protected per-strategy OOF: {mlp_smoke_artifacts.strategy_oof_path}')
else:
    print('MLP smoke is gated. Set RUN_MLP_SMOKE=True when ready.')

### Check smoke output before enabling MLP tuning

Tuning uses up to 100 epochs with patience 10 for its inner early-stopping run, followed by a fresh full-training refit for the selected epoch count. If many candidates reach the epoch limit, inspect training histories before deciding whether to extend it. Every candidate/fold can train two networks; candidate count is not the number of training runs. The standard checkpoint boundary remains a completed outer fold: an interruption during early stopping or refitting reruns that unfinished fold. There is no epoch-level restart.

Use the same code, dependencies, candidate parameters, training device, frozen splits, and checkpoint directory to resume. Changes to shared `classic.py` can invalidate earlier models' unfinished checkpoints too; preserve old runs and use a fresh directory for new experiments. Do not bypass manifest checks. Version the artifact suffix when changing the experiment. No Stage-3 rerun is required. Checkpoints survive only while their files remain available in the controlled environment.

GPU training is available when the installed PyTorch build and runtime expose CUDA. For insufficient GPU memory, reduce batch size and start a newly versioned experiment, or explicitly choose CPU. Model/OOF/checkpoint files are protected derivatives and must not be published or sent to external services.

In [ ]:
RUN_MLP_TUNING = False
mlp_tuning_result = None

if RUN_MLP_TUNING:
    mlp_tuning_result = train_mlp(
        static, splits, profile='tuning', device=MLP_DEVICE,
        checkpoint_dir=MLP_CHECKPOINT_DIR,
        resume=True, progress_callback=print,
    )
    mlp_tuning_artifacts = save_static_training_result(
        mlp_tuning_result, artifact_suffix=f'{MLP_ARTIFACT_SUFFIX}_tuning'
    )
    display(mlp_tuning_result.strategy_metrics)
    display(mlp_tuning_result.fold_metrics[[
        'candidate', 'fold', 'selected_epochs', 'selection_epochs',
        'inner_validation_auroc', 'fit_seconds',
    ]])
    print(f'Final MLP candidate: {mlp_tuning_result.best_candidate.name}')
    print(f'MLP protected OOF: {mlp_tuning_artifacts.oof_path}')
    print(f'MLP protected per-strategy OOF: {mlp_tuning_artifacts.strategy_oof_path}')
    print(f'MLP fitted development model: {mlp_tuning_artifacts.model_path}')
else:
    print('MLP tuning is gated. Set RUN_MLP_TUNING=True after checking smoke.')

In [ ]:
mlp_checked_result = (
    mlp_tuning_result if mlp_tuning_result is not None else mlp_smoke_result
)
if mlp_checked_result is not None:
    mlp_oof = mlp_checked_result.oof_predictions
    assert len(mlp_oof) == len(splits.dev_indices)
    assert mlp_oof['row_index'].is_unique
    assert set(mlp_oof['row_index']) == set(splits.dev_indices)
    assert set(mlp_oof['row_index']).isdisjoint(splits.test_indices)
    assert np.isfinite(mlp_oof['probability']).all()
    assert mlp_oof['probability'].between(0, 1).all()
    assert set(mlp_oof['cv_fold']) == set(range(N_CV_FOLDS))
    mlp_strategy_oof = mlp_checked_result.strategy_oof_predictions
    n_mlp_strategies = mlp_checked_result.strategy_metrics['strategy'].nunique()
    assert len(mlp_strategy_oof) == len(splits.dev_indices) * n_mlp_strategies
    assert not mlp_strategy_oof.duplicated(['strategy', 'row_index']).any()
    assert mlp_strategy_oof.groupby('strategy')['row_index'].nunique().eq(len(splits.dev_indices)).all()
    assert set(mlp_strategy_oof['row_index']) == set(splits.dev_indices)
    assert np.isfinite(mlp_strategy_oof['probability']).all()
    assert mlp_strategy_oof['probability'].between(0, 1).all()
    print(f'MLP global-best OOF rows verified: {len(mlp_oof):,}')
    print(f'MLP per-strategy OOF rows verified: {len(mlp_strategy_oof):,}')
    print('MLP internal test remains sealed: no Stage-4 predictions were created for it.')
else:
    print('MLP OOF audit skipped: no MLP run has completed in this session.')

## How the common framework is reused

`train_static_model` owns the invariant workflow: validate frozen splits → compute fold-local weights or resample with SMOTENC → cross-fit every candidate → calculate fold AUROC/AUPRC/Brier → retain the best candidate and OOF predictions for every strategy → refit the global winner on the complete development set. A later model supplies only: (1) an estimator factory, (2) candidate configurations, and (3) preprocessing/imbalance flags.

LR, XGBoost, Random Forest, SVM, and the PyTorch MLP share the training framework. SVM adds patient-grouped inner sigmoid calibration around freshly built fold-local pipelines. MLP uses a patient-grouped inner holdout to select its epoch count, then refits on the full current training partition. Both still expose `predict_proba()`. LR/XGBoost/RF retain their existing direct probability workflow. MLP's network and optimizer live in `src/models/mlp.py`; `classic.py` orchestrates its static-feature experiments, while `deep.py` remains reserved for the hourly LSTM. Additional imbalance sensitivity experiments and any XGBoost early stopping need separate, leakage-safe configuration.

## LR milestone acceptance

Before implementing the remaining models, confirm only aggregate outputs: the six-row imbalance screening table, each strategy's best candidate, five-fold AUROC/AUPRC/Brier summaries, calibration intercept/slope, predicted-risk prevalence, development OOF row counts, and saved artifact paths. Do not share OOF files or patient-level rows. F1 threshold selection remains deferred to Stage 5.